In [1]:
# 🛠️ 環境初始化：安裝缺失的依賴套件
%pip install -q roboflow nvidia-ml-py

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#從roboflow下載資料集
from roboflow import Roboflow
rf = Roboflow(api_key="yPijl9ttsvsSu6GsWCSz")
project = rf.workspace("ariess-workspace-rc756").project("strawberry-maturity-yolo-graduate")
dataset = project.version(1).download("yolov11")

In [ ]:
# 上傳模型到roboflow上
from roboflow import Roboflow

# 1. 初始化
rf = Roboflow(api_key="yPijl9ttsvsSu6GsWCSz")
project = rf.workspace("ariess-workspace-rc756").project("strawberry-maturity-yolo-graduate")

# 要部署到的版本號
version = project.version(2)

# 3. 上傳權重
print("🚀 正在上傳 YOLO11m 權重...")
# 注意：一個版本通常只能部署一個模型，如果你要同時存兩個，建議分別部署到不同版本，或只部署最強的一個
version.deploy(
    model_type="yolov11", 
    model_path="runs/detect/exp1b_yolo11m_baseline" #會自動找到weights\best.pt
)

print("✅ 部署指令已送出，請至 Roboflow 後台查看進度。")


In [ ]:
# 分析切分後的類別分佈
import os
import yaml
import pandas as pd

# --- 設定路徑 ---
# 指向指定的資料夾 (配合畢業專題 Roboflow 匯出路徑)
YOLO_DATA_DIR = 'strawberry-maturity-yolo-graduate-1' 
DATA_YAML_PATH = os.path.join(YOLO_DATA_DIR, 'data.yaml')

# --- 函數：讀取並統計標註 ---
def count_labels(label_dir, class_names):
    """
    遍歷指定資料夾內的 .txt 標註檔，並統計:
    1. 每個類別的物件數量
    2. 純背景圖片 (空標註檔) 的數量
    """
    counts = {name: 0 for name in class_names}
    total_files = 0
    total_objects = 0
    background_files = 0
    
    if not os.path.exists(label_dir):
        print(f"[WARN] 找不到路徑 {label_dir}。跳過此分割。")
        return counts, 0, 0, 0

    for filename in os.listdir(label_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(label_dir, filename)
            total_files += 1
            
            try:
                # 檢查是否為空檔案 (0 bytes)
                if os.path.getsize(file_path) == 0:
                    background_files += 1
                    continue

                has_objects = False
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        
                        has_objects = True 
                        
                        try:
                            # YOLO 格式: <class_id> <x> <y> <w> <h>
                            class_id_str = line.split()[0]
                            class_index = int(class_id_str)
                            
                            if 0 <= class_index < len(class_names):
                                class_name = class_names[class_index]
                                counts[class_name] += 1
                                total_objects += 1
                        except (ValueError, IndexError):
                            continue 
                
                if not has_objects:
                    background_files += 1
                                    
            except Exception as e:
                print(f"[ERROR] 讀取標註檔 {filename} 失敗: {e}")
                continue

    return counts, total_files, total_objects, background_files


# --- 主程式流程 ---
def analyze_split_distribution():
    # 1. 載入類別名稱
    try:
        # 確保以 utf-8 讀取，避免中文字元問題
        with open(DATA_YAML_PATH, 'r', encoding='utf-8') as f:
            data_yaml = yaml.safe_load(f)
        
        class_names = data_yaml.get('names')
        if not class_names:
            raise ValueError("在 data.yaml 中找不到 'names' 列表。")
        
        print(f"[OK] 成功讀取 {len(class_names)} 個類別: {class_names}")
    
    except FileNotFoundError:
        print(f"[ERROR] 找不到 data.yaml 檔案: {DATA_YAML_PATH}")
        return
    except Exception as e:
        print(f"[ERROR] 解析 data.yaml 失敗: {e}")
        return

    # 2. 統計各個分割集 (配合 Roboflow 結構將 'val' 改為 'valid')
    splits = ['train', 'valid', 'test']
    results = {}

    for s in splits:
        label_dir = os.path.join(YOLO_DATA_DIR, s, 'labels')
        counts, files, objects, bg = count_labels(label_dir, class_names)
        results[s] = {
            'counts': counts,
            'files': files,
            'objects': objects,
            'bg': bg
        }

    # 3. 整理資料夾物件分佈表格
    df_data = {
        'Train Objects': results['train']['counts'],
        'Valid Objects': results['valid']['counts'],
        'Test Objects': results['test']['counts']
    }
    df = pd.DataFrame(df_data).T 
    df['Total Objects'] = df.sum(axis=1)
    
    # 4. 輸出總結
    print("\n" + "="*60)
    print("--- 訓練 / 驗證 / 測試資料集統計總結 ---")
    print("="*60)
    
    for s in splits:
        icon = "[T]" if s == 'train' else ("[V]" if s == 'valid' else "[E]")
        name = s.capitalize()
        res = results[s]
        print(f"{icon} {name} Set:")
        print(f"   - 總圖片數:      {res['files']}")
        print(f"   - 有物件圖片:    {res['files'] - res['bg']}")
        print(f"   - 純背景圖片:    {res['bg']} (負樣本)")
        print(f"   - 標註物件總數:  {res['objects']}")
        print("-" * 40)
    
    # 輸出詳細的類別分佈表
    print("\n[INFO] 類別物件數量分佈 (Object Counts)")
    print(df.to_string())

    # 5. 檢查分佈平衡性 (比例分析)
    print("\n[INFO] 平衡性檢查 (以 Train 為基準的比例)")
    print(f"{'Class Name':<15} | {'Train:Valid':<12} | {'Train:Test':<10}")
    print("-" * 48)

    def get_ratio_str(train_val, target_val):
        if target_val == 0:
            return "Inf" if train_val > 0 else "0.0"
        return f"{train_val / target_val:.1f}"

    # 背景圖比例
    bg_v = get_ratio_str(results['train']['bg'], results['valid']['bg'])
    bg_t = get_ratio_str(results['train']['bg'], results['test']['bg'])
    print(f"{'[Background]':<15} | {bg_v:<12} | {bg_t:<10}")

    for class_name in class_names:
        tr_c = results['train']['counts'].get(class_name, 0)
        va_c = results['valid']['counts'].get(class_name, 0)
        te_c = results['test']['counts'].get(class_name, 0)
        
        ratio_v = get_ratio_str(tr_c, va_c)
        ratio_t = get_ratio_str(tr_c, te_c)

        print(f"{class_name:<15} | {ratio_v:<12} | {ratio_t:<10}")
    
    print("\n(註：若比例設定為 8:1:1，建議比例約為 8.0)")     
    print("="*60)


if __name__ == "__main__":
    analyze_split_distribution()


In [ ]:
#監控訓練中的最新結果
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import glob
import threading
from IPython.display import clear_output, display

# 旗標：控制背景監控是否繼續
keep_running = True

def get_latest_results_csv(base_path="Strawberry_YOLOv11_4060ti-16G/*/results.csv"):
    files = glob.glob(base_path)
    if not files: return None
    return max(files, key=os.path.getmtime)

def plot_training_results():
    csv_path = get_latest_results_csv()
    if csv_path is None or not os.path.exists(csv_path):
        print(f"⌛ 等待訓練數據產生中...")
        return

    try:
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
    except: return

    if df.empty: return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # --- Loss 圖 ---
    ax1.plot(df['epoch'], df['train/box_loss'], label='Train Box', color='blue', alpha=0.4)
    ax1.plot(df['epoch'], df['val/box_loss'], label='Val Box', color='blue', linestyle='--')
    ax1.plot(df['epoch'], df['train/cls_loss'], label='Train Cls', color='orange', alpha=0.4)
    ax1.plot(df['epoch'], df['val/cls_loss'], label='Val Cls', color='orange', linestyle='--')
    ax1.set_title(f"Loss Curves ({os.path.basename(os.path.dirname(csv_path))})")
    ax1.legend(); ax1.grid(True)

    # --- mAP 圖 ---
    ax2.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', color='green', linewidth=2)
    ax2.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', color='red', linewidth=1)
    ax2.set_title("mAP Metrics")
    ax2.legend(); ax2.grid(True)

    clear_output(wait=True)
    plt.tight_layout()
    plt.show()
    print(f"📊 最新更新時間: {time.strftime('%Y-%m-%d %H:%M:%S')} | 數據源: {csv_path}")

def monitor_loop(interval=10):
    while keep_running:
        plot_training_results()
        time.sleep(interval)

# 啟動背景監控執行緒
monitor_thread = threading.Thread(target=monitor_loop, args=(10,), daemon=True)
monitor_thread.start()
print("🟢 已經在背景啟動訓練結果即時監控！每一輪(Epoch)結束後圖表將會自動重繪。")


In [ ]:
# 分析最新訓練結果
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# --- 設定：訓練結果的根目錄 ---
# 通常是 'runs/detect'，如果您有更改 project 參數，請修改這裡
RUNS_DIR = 'runs/detect'

def find_latest_train_dir(base_dir):
    """自動尋找最新的訓練資料夾 (例如 train, train2, train12...)"""
    # 搜尋所有以 train 開頭的資料夾
    search_path = os.path.join(base_dir, '*')
    dirs = glob.glob(search_path)
    
    if not dirs:
        return None
    
    # 根據修改時間排序，最新的在最後
    latest_dir = max(dirs, key=os.path.getmtime)
    return latest_dir

def analyze_training_results():
    # 1. 找到最新的訓練資料夾
    latest_dir = find_latest_train_dir(RUNS_DIR)
    
    if latest_dir is None:
        print(f"❌ 錯誤：在 '{RUNS_DIR}' 中找不到任何訓練資料夾。")
        return

    csv_path = os.path.join(latest_dir, 'results.csv')
    print(f"📂 正在讀取最新的訓練結果：{csv_path}")
    
    if not os.path.exists(csv_path):
        print("❌ 錯誤：找不到 results.csv 檔案。")
        return

    # 2. 讀取 CSV
    try:
        df = pd.read_csv(csv_path)
        # 清理欄位名稱 (移除多餘空格)
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"❌ 讀取 CSV 失敗: {e}")
        return

    # 3. 找出最佳模型 (根據 mAP50-95)
    # Ultralytics 的欄位名稱通常是: 
    # metrics/precision(B), metrics/recall(B), metrics/mAP50(B), metrics/mAP50-95(B)
    
    target_metric = 'metrics/mAP50-95(B)'
    
    if target_metric not in df.columns:
        print(f"⚠️ 警告：找不到 '{target_metric}' 欄位，嘗試列印所有欄位：")
        print(df.columns.tolist())
        return

    # 找到數值最大的那一列
    best_idx = df[target_metric].idxmax()
    best_epoch_data = df.loc[best_idx]
    
    # 4. 顯示結果
    print("\n" + "="*50)
    print("🏆 最佳模型效能報告 (Best Model Performance)")
    print("="*50)
    print(f"📍 發生在 Epoch: {int(best_epoch_data['epoch'])}")
    print("-" * 50)
    print(f"🎯 mAP50-95 (綜合指標):  {best_epoch_data['metrics/mAP50-95(B)']:.4f}  <-- 主要參考")
    print(f"🥇 mAP50    (偵測指標):  {best_epoch_data['metrics/mAP50(B)']:.4f}")
    print("-" * 50)
    print(f"✅ Precision (準確率):    {best_epoch_data['metrics/precision(B)']:.4f}")
    print(f"🔍 Recall    (召回率):    {best_epoch_data['metrics/recall(B)']:.4f}")
    print("="*50)


# --- 執行分析 ---
analyze_training_results()

In [2]:
import sys, os
sys.argv = [sys.argv[0]]

# 確保本地 lib 資料夾在搜尋路徑中 (修復 conda 環境權限問題)
local_lib = os.path.abspath(os.path.join(os.getcwd(), 'agent_tools', 'lib'))
if local_lib not in sys.path:
    sys.path.insert(0, local_lib)

import pynvml
import torch

def safe_decode(val):
    if isinstance(val, bytes): return val.decode('utf-8', errors='ignore')
    return str(val)

def get_gpu_health_check():
    print("========================================================================")
    print("🚀 RTX 4060 Ti (16G) GPU 健檢程式 - 鎖定 pytorch 核心環境與 nvidia-ml-py 驅動")
    print("========================================================================")
    
    # 1. 檢查 PyTorch CUDA 可用度
    cuda_available = torch.cuda.is_available()
    print(f"🔹 PyTorch CUDA 可用狀態: {cuda_available}")
    if cuda_available:
        print(f"   - 顯示卡型號       : {torch.cuda.get_device_name(0)}")
        print(f"   - CUDA 驅動版本    : {torch.version.cuda}")
        print(f"   - PyTorch 核心版本 : {torch.__version__}")
    else:
        print("   ❌ PyTorch 無法辨識 GPU，請檢查 CUDA / PyTorch 版本對齊狀態！")
        return
        
    # 2. 透過 nvidia-ml-py 初始化並提取精確物理硬體數據
    print("\n🔹 nvidia-ml-py (NVML) 硬體即時數據診斷:")
    try:
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        
        # 取得各種資訊
        gpu_name = safe_decode(pynvml.nvmlDeviceGetName(handle))
        driver_ver = safe_decode(pynvml.nvmlSystemGetDriverVersion())
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        
        # 時脈與頻寬相關
        curr_graphics_clock = pynvml.nvmlDeviceGetClockInfo(handle, pynvml.NVML_CLOCK_GRAPHICS)
        curr_mem_clock = pynvml.nvmlDeviceGetClockInfo(handle, pynvml.NVML_CLOCK_MEM)
        max_graphics_clock = pynvml.nvmlDeviceGetMaxClockInfo(handle, pynvml.NVML_CLOCK_GRAPHICS)
        
        # 溫度與電力
        temp = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
        power_usage = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0 # 轉為 W
        power_limit = pynvml.nvmlDeviceGetPowerManagementLimit(handle) / 1000.0

        print(f"   - 正式名稱       : {gpu_name}")
        print(f"   - 驅動程式版本   : {driver_ver}")
        print(f"   - 總顯存 (VRAM)  : {mem_info.total / 1024**2:.0f} MB")
        print(f"   - 目前顯存佔用   : {mem_info.used / 1024**2:.0f} MB ({(mem_info.used/mem_info.total)*100:.1f}%)")
        print(f"   - 核心時脈       : {curr_graphics_clock} MHz (Max: {max_graphics_clock} MHz)")
        print(f"   - 記憶體時脈     : {curr_mem_clock} MHz")
        print(f"   - 當前溫度       : {temp} °C")
        print(f"   - 當前功耗       : {power_usage:.1f} W / {power_limit:.1f} W")
        
        # 計算理論頻寬
        pcie_gen = pynvml.nvmlDeviceGetMaxPcieLinkGeneration(handle)
        pcie_width = pynvml.nvmlDeviceGetMaxPcieLinkWidth(handle)
        print(f"   - PCIe 介面      : Gen {pcie_gen} x{pcie_width}")

    except Exception as e:
        print(f"   ❌ NVML 診斷失敗: {e}")
    finally:
        try: pynvml.nvmlShutdown()
        except: pass

    print(f"\n{'='*80}")
    print("✅ 健檢完成！目前環境與 RTX 4060 Ti 處於最佳狀態。")

get_gpu_health_check()


🚀 RTX 4060 Ti (16G) GPU 健檢程式 - 鎖定 pytorch 核心環境與 nvidia-ml-py 驅動
🔹 PyTorch CUDA 可用狀態: True
   - 顯示卡型號       : NVIDIA GeForce RTX 4060 Ti
   - CUDA 驅動版本    : 12.1
   - PyTorch 核心版本 : 2.5.1+cu121

🔹 nvidia-ml-py (NVML) 硬體即時數據診斷:
   - 正式名稱       : NVIDIA GeForce RTX 4060 Ti
   - 驅動程式版本   : 581.57
   - 總顯存 (VRAM)  : 16380 MB
   - 目前顯存佔用   : 2667 MB (16.3%)
   - 核心時脈       : 210 MHz (Max: 3105 MHz)
   - 記憶體時脈     : 405 MHz
   - 當前溫度       : 42 °C
   - 當前功耗       : 12.3 W / 165.0 W
   - PCIe 介面      : Gen 4 x8

✅ 健檢完成！目前環境與 RTX 4060 Ti 處於最佳狀態。


In [ ]:
# 分析多個訓練結果，找出最佳模型的效能指標與參數
import sys
import os
import pandas as pd
import yaml
import datetime
from glob import glob

sys.argv = [sys.argv[0]]
try:
    if 'ROOT_DIR' not in locals():
        ROOT_DIR = 'runs/detect'
except NameError:
    ROOT_DIR = 'runs/detect'

def make_path_relative(path):
    """將絕對路徑轉換為相對於目前專案的相對路徑，並處理異機搬移的路徑問題"""
    if not isinstance(path, str) or not path:
        return path
    
    # 統一斜線格式
    path = path.replace('\\', '/')
    current_workspace = os.getcwd().replace('\\', '/')
    
    # 1. 如果是在目前的專案路徑下，直接轉相對路徑
    if path.lower().startswith(current_workspace.lower()):
        try:
            rel = os.path.relpath(path, current_workspace)
            return rel.replace('\\', '/')
        except:
            pass
            
    # 2. 如果是絕對路徑但不在目前的專案下，嘗試關鍵字修復 (例如從桌面搬過來的)
    if os.path.isabs(path):
        # 尋找路徑中的專案關鍵資料夾
        keywords = ['runs', 'strawberry-maturity-yolo-graduate-1', 'yolo_data', 'best_weights']
        for kw in keywords:
            if kw in path:
                parts = path.split('/')
                try:
                    idx = parts.index(kw)
                    rel_candidate = '/'.join(parts[idx:])
                    if os.path.exists(os.path.join(current_workspace, rel_candidate)):
                        return rel_candidate
                    return rel_candidate # 即使不存在也回傳相對化的名稱，保持視覺整潔
                except ValueError:
                    continue
        
        # 3. 如果只是單純檔案且在專案根目錄存在
        basename = os.path.basename(path)
        if os.path.exists(os.path.join(current_workspace, basename)):
            return basename

    return path

def get_data_count(data_yaml_path):
    """從 data.yaml 獲取訓練集圖片數量，增強路徑自癒能力"""
    if not data_yaml_path:
        return "N/A"
    
    current_workspace = os.getcwd()
    
    # 先將路徑相對化
    data_yaml_path = make_path_relative(data_yaml_path)
    
    if not os.path.exists(data_yaml_path):
        potential_names = ['strawberry-maturity-yolo-graduate-1', 'yolo_data', 'roboflow_側拍']
        for name in potential_names:
            if name in data_yaml_path:
                alt_path = os.path.join(current_workspace, name, 'data.yaml')
                if os.path.exists(alt_path):
                    data_yaml_path = alt_path
                    break
    
    if not os.path.exists(data_yaml_path):
        basename = os.path.basename(data_yaml_path)
        if os.path.exists(basename):
            data_yaml_path = basename

    if not os.path.exists(data_yaml_path):
        return "N/A"
        
    try:
        with open(data_yaml_path, 'r', encoding='utf-8') as f:
            d_yaml = yaml.safe_load(f)
            yaml_dir = os.path.dirname(os.path.abspath(data_yaml_path))
            base_path = d_yaml.get('path', '')
            train_rel = d_yaml.get('train', '')
            
            if os.path.isabs(train_rel):
                full_path = train_rel
            elif base_path:
                if os.path.isabs(base_path):
                    if not os.path.exists(base_path):
                        folder_name = os.path.basename(base_path)
                        full_path = os.path.join(current_workspace, folder_name, train_rel)
                    else:
                        full_path = os.path.join(base_path, train_rel)
                else:
                    full_path = os.path.join(yaml_dir, base_path, train_rel)
            else:
                full_path = os.path.join(yaml_dir, train_rel)
            
            full_path = os.path.normpath(full_path)
            
            if not os.path.exists(full_path):
                clean_rel = train_rel.replace('../', '').replace('./', '')
                alt_full_path = os.path.join(yaml_dir, clean_rel)
                if os.path.exists(alt_full_path):
                    full_path = alt_full_path

            if os.path.exists(full_path):
                imgs = [f for f in os.listdir(full_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                return len(imgs)
    except:
        pass
    return "N/A"

def format_duration(seconds):
    """將秒數轉為易讀格式"""
    if seconds <= 0: return "< 1m"
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    if h > 0: return f"{h}h {m}m"
    return f"{m}m"

def compare_training_results():
    """
    從各訓練目錄提取效能、參數、日期、時長與資料量進行排名
    """
    print(f"🔍 正在掃描 {ROOT_DIR} 下的所有訓練紀錄...")
    
    summary_data = []
    search_path = os.path.join(ROOT_DIR, 'exp*')
    folders = glob(search_path)
    
    if not folders:
        print("❌ 找不到任何訓練資料夾，請確認路徑是否正確。")
        return

    for folder in folders:
        if not os.path.isdir(folder): continue
        
        csv_path = os.path.join(folder, 'results.csv')
        args_path = os.path.join(folder, 'args.yaml')
        
        train_date = "N/A"
        mtime_duration = 0
        if os.path.exists(args_path):
            t_start = os.path.getmtime(args_path)
            train_date = datetime.datetime.fromtimestamp(t_start).strftime('%Y-%m-%d')
            if os.path.exists(csv_path):
                t_end = os.path.getmtime(csv_path)
                mtime_duration = t_end - t_start
        
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                df.columns = [c.strip() for c in df.columns]
                
                if 'time' in df.columns and not df['time'].isnull().all():
                    total_seconds = df['time'].iloc[-1]
                    duration_str = format_duration(total_seconds)
                else:
                    duration_str = format_duration(mtime_duration)
                
                map_cols = [c for c in df.columns if 'mAP50-95' in c]
                target_col = map_cols[0] if map_cols else None
                
                if target_col and not df[target_col].isnull().all():
                    best_idx = df[target_col].idxmax()
                    best_row = df.iloc[best_idx]
                    
                    params = {}
                    data_count = "N/A"
                    if os.path.exists(args_path):
                        with open(args_path, 'r', encoding='utf-8') as f:
                            args_data = yaml.safe_load(f)
                            # 轉換模型路徑為相對路徑
                            params['Model'] = make_path_relative(args_data.get('model', 'Unknown'))
                            params['Total_Epochs'] = args_data.get('epochs', 0)
                            params['Patience'] = args_data.get('patience', 0)
                            params['Optimizer'] = args_data.get('optimizer', 'auto')
                            params['Batch'] = args_data.get('batch', 0)
                            data_count = get_data_count(args_data.get('data'))
                    else:
                        params = {'Model': 'N/A', 'Total_Epochs': 'N/A', 'Patience': 'N/A', 'Optimizer': 'N/A', 'Batch': 'N/A'}

                    folder_name = os.path.basename(folder)
                    summary_data.append({
                        '資料夾': folder_name,
                        '日期': train_date,
                        '時長': duration_str,
                        '資料量': data_count,
                        '模型': params['Model'],
                        '優化器': params['Optimizer'],
                        'Batch': params['Batch'],
                        '總回合': params['Total_Epochs'],
                        '實際回合': len(df),
                        '早停': params['Patience'],
                        'mAP50-95': best_row[target_col],
                        'mAP50': best_row.get('metrics/mAP50(B)', best_row.get('metrics/mAP50', 0)),
                        'Precision': best_row.get('metrics/precision(B)', best_row.get('metrics/precision', 0)),
                        'Recall': best_row.get('metrics/recall(B)', best_row.get('metrics/recall', 0))
                    })
            except Exception as e:
                print(f"⚠️ 無法讀取 {folder}: {e}")

    if summary_data:
        result_df = pd.DataFrame(summary_data)
        result_df = result_df.sort_values(by=['日期', 'mAP50-95'], ascending=[False, False]).reset_index(drop=True)
        
        print("=" * 140)
        print("🏆 訓練效能與參數總和排行榜 (依照日期由新到舊排序)")
        print("=" * 140)
        try: print(result_df.to_markdown(index=False, numalign='left', stralign='left'))
        except: print(result_df.to_string(index=False))
        print("=" * 140)
        
        best_overall = result_df.loc[result_df['mAP50-95'].idxmax()]
        print(f"🥇 歷史表現最優異訓練: {best_overall['資料夾']} (mAP50-95: {best_overall['mAP50-95']:.4f}, 日期: {best_overall['日期']})")
        print(f"💡 推薦使用此權重進行部署: runs/detect/{best_overall['資料夾']}/weights/best.pt")
    else:
        print("ℹ️ 目前尚無訓練紀錄。")

if __name__ == "__main__":
    compare_training_results()


In [ ]:
#印出各項圖表
import os
import glob
from IPython.display import display, Image, Markdown

# --- 設定：訓練結果目錄 ---
RUNS_DIR = 'runs\detect'

def find_latest_train_dir(base_dir):
    """自動尋找最新的訓練資料夾"""
    search_path = os.path.join(base_dir, '*')
    dirs = glob.glob(search_path)
    if not dirs:
        return None
    # 根據修改時間排序，最新的在最後
    return max(dirs, key=os.path.getmtime)

def show_yolo_plots():
    latest_dir = find_latest_train_dir(RUNS_DIR)
    
    if not latest_dir:
        print(f"❌ 找不到訓練資料夾於 {RUNS_DIR}")
        return

    print(f"📂 正在展示來自 [ {latest_dir} ] 的結果圖表\n")

    # 想要顯示的圖表清單 (依照優先順序)
    images_to_show = {
        "results.png": "📈 訓練過程總覽 (損失 & mAP 曲線)",
        "confusion_matrix_normalized.png": "🟦 混淆矩陣 (正規化) - 查看類別混淆情況",
        "confusion_matrix.png": "🟦 混淆矩陣 (原始數量)",
        "F1_curve.png": "🎯 F1-Score 曲線 (準確與召回的平衡)",
        "PR_curve.png": "📉 Precision-Recall 曲線",
        "val_batch0_pred.jpg": "🖼️ 驗證集預測範例 (Batch 0) - 模型實際看到的樣子",
        "val_batch0_labels.jpg": "🏷️ 驗證集真實標籤 (Batch 0) - 正確答案",
        "val_batch1_pred.jpg": "🖼️ 驗證集預測範例 (Batch 1)",
    }

    found_any = False
    for filename, description in images_to_show.items():
        img_path = os.path.join(latest_dir, filename)
        
        if os.path.exists(img_path):
            found_any = True
            # 使用 Markdown 顯示標題，讓排版好看一點
            display(Markdown(f"### {description}"))
            display(Image(filename=img_path, width=800)) # width 可自行調整
            print("\n" + "-"*60 + "\n")
    
    if not found_any:
        print("⚠️ 警告：在這個資料夾中找不到上述任何一張圖表。請確認訓練是否已完成至少一個 Epoch。")

# --- 執行 ---
show_yolo_plots()

In [ ]:
#排名各次訓練結果
import os
import pandas as pd
from glob import glob

# 設定您的訓練結果路徑
ROOT_DIR = 'runs\detect'

def compare_training_results():
    print(f"🔍 正在掃描 {ROOT_DIR} 下的所有訓練紀錄...\n")
    
    summary_data = []
    
    # 搜尋所有 train 開頭的資料夾
    search_path = os.path.join(ROOT_DIR, '*')
    folders = glob(search_path)
    
    if not folders:
        print("❌ 找不到任何訓練資料夾，請確認路徑是否正確。")
        return

    for folder in folders:
        csv_path = os.path.join(folder, 'results.csv')
        
        # 檢查是否存在 results.csv
        if os.path.exists(csv_path):
            try:
                # 讀取 CSV
                df = pd.read_csv(csv_path)
                
                # 清理欄位名稱 (YOLO 的 CSV 欄位名稱常帶有空格)
                df.columns = [c.strip() for c in df.columns]
                
                # 找出 mAP50-95 最高的那一列 (代表該次訓練的最佳表現)
                # 欄位通常是 'metrics/mAP50-95(B)'
                best_idx = df['metrics/mAP50-95(B)'].idxmax()
                best_epoch = df.iloc[best_idx]
                
                folder_name = os.path.basename(folder)
                
                summary_data.append({
                    'Folder (資料夾)': folder_name,
                    'Epochs (總輪數)': len(df),
                    'Best mAP50-95': best_epoch['metrics/mAP50-95(B)'], # 綜合指標 (最重要)
                    'Best mAP50': best_epoch['metrics/mAP50(B)'],       # 偵測成功率
                    'Best Precision': best_epoch['metrics/precision(B)'],
                    'Best Recall': best_epoch['metrics/recall(B)']
                })
            except Exception as e:
                print(f"⚠️ 無法讀取 {folder}: {e}")

    # 轉為 DataFrame 並排序
    if summary_data:
        result_df = pd.read_csv(csv_path) # 修正: 這裡不需要重讀，直接用 summary_data 建立 DataFrame
        result_df = pd.DataFrame(summary_data)
        
        # 依照 mAP50-95 由高到低排序
        result_df = result_df.sort_values(by='Best mAP50-95', ascending=False).reset_index(drop=True)
        
        print("-" * 80)
        print("🏆 訓練結果排行榜 (依照 mAP50-95 排序)")
        print("-" * 80)
        print(result_df.to_string(index=False))
        print("-" * 80)
        
        # 告訴使用者最好的模型在哪
        best_run = result_df.iloc[0]
        print(f"\n✅ 最強模型位於: {best_run['Folder (資料夾)']}")
        print(f"   mAP50-95: {best_run['Best mAP50-95']:.4f}")
        print(f"   mAP50   : {best_run['Best mAP50']:.4f}")
        print(f"\n💡 請使用此權重檔進行口試展示: runs/detect/{best_run['Folder (資料夾)']}/weights/best.pt")
        
    else:
        print("❌ 沒有讀取到有效的訓練數據。")

if __name__ == '__main__':
    compare_training_results()

In [ ]:

# ==========================================
# 🔍 全資料集自動診斷 (Roboflow & runs/detect 結構版)
# ==========================================
import os, glob, cv2, datetime, re, shutil, json, torch
import pandas as pd
from ultralytics import YOLO
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from tqdm.notebook import tqdm


# 1. 設定目標模型 ( '*' 代表自動抓取 runs/detect 裡最新的一個)
TARGET_TRAIN_ID = '*'      
# 2. 資料集根目錄
DATASET_ROOT = 'strawberry-maturity-yolo-graduate-1' 
# 3. 診斷結果存放處
BASE_SAVE_DIR = '診斷結果'
CONF_LEVEL = 0.5
IOU_THRESHOLD = 0.45

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def draw_chinese_text(img, text, position, color=(0, 0, 255), size=20):
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    font_path = "C:\\Windows\\Fonts\\msjh.ttc" # Windows 微軟正黑體
    font = ImageFont.truetype(font_path, size) if os.path.exists(font_path) else ImageFont.load_default()
    draw.text(position, text, font=font, fill=color[::-1])
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

def calculate_iou(box1, box2):
    x1, y1, x2, y2 = max(box1[0], box2[0]), max(box1[1], box2[1]), min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter / (area1 + area2 - inter + 1e-6)

def find_target_model(target_id):
    """ 自動尋找權重檔，支援 '*' 自動抓取最新訓練 """
    base_path = 'runs/detect'
    if target_id == '*':
        # 抓取所有 exp 資料夾，包含我們的 exp1a, exp1b...
        dirs = glob.glob(os.path.join(base_path, 'exp*'))
        if not dirs: 
            # 如果正式目錄還沒搬移，檢查一下有沒有正在訓練的中繼站
            if os.path.exists('Strawberry_YOLOv11'):
                dirs = glob.glob(os.path.join('Strawberry_YOLOv11', 'exp*'))
        
        if not dirs: raise FileNotFoundError("❌ 找不到任何訓練目錄。")
        latest_dir = max(dirs, key=os.path.getmtime)
        path = os.path.join(latest_dir, 'weights', 'best.pt')
    else:
        # 指定 ID 或名稱
        target_dir = os.path.join(base_path, target_id)
        path = os.path.join(target_dir, 'weights', 'best.pt')
        
    if os.path.exists(path): 
        print(f"🎯 已鎖定權重檔: {path}")
        return path
    else: 
        raise FileNotFoundError(f"❌ 找不到權重檔: {path}")

def run_full_diagnostic():
    model_path = find_target_model(TARGET_TRAIN_ID)
    train_dir = os.path.dirname(os.path.dirname(model_path))
    model = YOLO(model_path)
    
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id = os.path.basename(train_dir)
    run_save_dir = os.path.join(BASE_SAVE_DIR, f"診斷_{run_id}_{timestamp}")
    os.makedirs(run_save_dir, exist_ok=True)
    
    performance_info = {"model_path": model_path, "train_run": run_id}
    
    # --- 1. 提取訓練成效 (results.csv) ---
    results_csv = os.path.join(train_dir, 'results.csv')
    if os.path.exists(results_csv):
        try:
            with open(results_csv, 'r', encoding='utf-8') as f:
                df = pd.read_csv(f)
            df.columns = [c.strip() for c in df.columns]
            performance_info['metrics'] = df.iloc[-1].to_dict()
            print(f"📈 成功提取結果：mAP50 = {df['metrics/mAP50(B)'].iloc[-1]:.4f}")
        except: pass

    # --- 2. 收集所有圖片路徑 (train/val/test) ---
    all_images = []
    splits = ['train', 'val', 'test']
    for split in splits:
        split_img_dir = os.path.join(DATASET_ROOT, split, 'images')
        if not os.path.exists(split_img_dir): continue
        for ext in ['*.jpg', '*.png', '*.jpeg']:
            all_images.extend(glob.glob(os.path.join(split_img_dir, ext)))
            
    all_images = sorted(list(set(all_images)), key=lambda x: natural_sort_key(os.path.basename(x)))
    print(f"📦 資料集載入完成：共發現 {len(all_images)} 張圖片 (含 train/val/test)")

    # --- 3. 開始診斷 ---
    stats = { "total": 0, "ok": 0, "missed": 0, "extra": 0, "wrong_cls": 0 }
    report_file = os.path.join(run_save_dir, "全資料集診斷報告.txt")
    
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write(f"=== 草莓資料集全量診斷報告 ({run_id}) ===\n")
        f.write(f"診斷時間: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("-" * 50 + "\n")

        for img_p in tqdm(all_images, desc="診斷進度"):
            stats["total"] += 1
            base_name = os.path.basename(img_p)
            
            # 推論
            results = model.predict(img_p, conf=CONF_LEVEL, imgsz=640, verbose=False)
            pred_boxes = results[0].boxes.xyxy.cpu().numpy()
            pred_cls = results[0].boxes.cls.cpu().numpy().astype(int)
            
            # 尋找對應的標註檔 (Roboflow 結構: images/.. -> labels/..)
            split_dir = os.path.dirname(os.path.dirname(img_p))
            label_p = os.path.join(split_dir, 'labels', os.path.splitext(base_name)[0] + ".txt")
            
            gt_data = []
            if os.path.exists(label_p):
                img_data = cv2.imread(img_p)
                if img_data is not None:
                    h, w = img_data.shape[:2]
                    with open(label_p, 'r') as tf:
                        for line in tf.readlines():
                            pts = line.split()
                            if len(pts) >= 5:
                                c, x, y, nw, nh = map(float, pts[:5])
                                gt_data.append((int(c), (x-nw/2)*w, (y-nh/2)*h, (x+nw/2)*w, (y+nh/2)*h))
            
            # 比對邏輯
            matched_gt = [False] * len(gt_data)
            matched_pred = [False] * len(pred_boxes)
            img_errs = {"missed": [], "extra": [], "wrong": []}
            
            for i, gt in enumerate(gt_data):
                for j, pred in enumerate(pred_boxes):
                    if calculate_iou(gt[1:], pred) > IOU_THRESHOLD:
                        matched_gt[i] = True
                        matched_pred[j] = True
                        if gt[0] != pred_cls[j]:
                            img_errs["wrong"].append((gt, pred_cls[j]))
            
            for i, m in enumerate(matched_gt): 
                if not m: img_errs["missed"].append(gt_data[i])
            for j, m in enumerate(matched_pred): 
                if not m: img_errs["extra"].append(pred_boxes[j])
            
            has_issue = (len(img_errs["missed"]) + len(img_errs["extra"]) + len(img_errs["wrong"])) > 0
            if has_issue:
                stats["missed"] += len(img_errs["missed"])
                stats["extra"] += len(img_errs["extra"])
                stats["wrong_cls"] += len(img_errs["wrong"])
                
                # 繪製錯誤圖
                diag_img = cv2.imread(img_p)
                for gt in img_errs["missed"]: # 漏抓用紅色
                    cv2.rectangle(diag_img, (int(gt[1]), int(gt[2])), (int(gt[3]), int(gt[4])), (0, 0, 255), 3)
                    diag_img = draw_chinese_text(diag_img, "漏抓", (int(gt[1]), int(gt[2])-25), color=(0, 0, 255))
                for gt, p_c in img_errs["wrong"]: # 類別錯用黃色
                    cv2.rectangle(diag_img, (int(gt[1]), int(gt[2])), (int(gt[3]), int(gt[4])), (0, 255, 255), 3)
                    diag_img = draw_chinese_text(diag_img, f"類別錯:應為{model.names[gt[0]]}", (int(gt[1]), int(gt[2])-25), color=(0, 255, 255))
                for p_box in img_errs["extra"]: # 多抓用橘色
                    cv2.rectangle(diag_img, (int(p_box[0]), int(p_box[1])), (int(p_box[2]), int(p_box[3])), (0, 165, 255), 3)
                    diag_img = draw_chinese_text(diag_img, "多抓/漏標", (int(p_box[0]), int(p_box[1])-25), color=(0, 165, 255))
                
                # 儲存錯誤圖片
                cv2.imwrite(os.path.join(run_save_dir, f"ERR_{base_name}"), diag_img)
                f.write(f"[ISSUE] {base_name} | 漏:{len(img_errs['missed'])} | 多:{len(img_errs['extra'])} | 誤:{len(img_errs['wrong'])}\n")
            else:
                stats["ok"] += 1
                f.write(f"[OK]    {base_name}\n")
        
        # 寫入最終統計
        f.write("-" * 50 + "\n")
        f.write(f"總張數: {stats['total']} | 完美張數: {stats['ok']} | 錯誤張數: {stats['total']-stats['ok']}\n")
        f.write(f"漏抓總數: {stats['missed']} | 多抓總數: {stats['extra']} | 類別錯誤: {stats['wrong_cls']}\n")
            
    print(f"✅ 診斷完成！結果儲存至: {run_save_dir}")

# 執行
run_full_diagnostic()
